# ROGII Wellbore Geology

## Score: 19.309

In [ ]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

TRAIN_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")
SAMPLE_SUB = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv")

ROLL_WINDOWS = (5, 15, 31)
TW_LOCAL_WINDOWS = (15, 31)
LGB_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}
XGB_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.05,
    "max_depth": 8,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "lambda": 1.0,
    "tree_method": "hist",
    "seed": 42,
}
CAT_PARAMS = {
    "loss_function": "RMSE",
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 3.0,
    "random_seed": 42,
    "verbose": False,
}
NUM_BOOST_ROUND = 2000
EARLY_STOPPING = 100
N_FOLDS = 5
MODEL_NAMES = ("lgb", "xgb", "cat")

In [ ]:
def list_wells(data_dir: Path) -> list[str]:
    wells = sorted({p.name.split("__")[0] for p in data_dir.glob("*__horizontal_well.csv")})
    return wells


def load_horizontal(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__horizontal_well.csv"
    df = pd.read_csv(path)
    df.insert(0, "well", well)
    df["row_idx"] = np.arange(len(df))
    return df


def load_typewell(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__typewell.csv"
    tw = pd.read_csv(path).sort_values("TVT").reset_index(drop=True)
    return tw


def interp_tw(tvt_values: np.ndarray, tw_tvt: np.ndarray, tw_vals: np.ndarray) -> np.ndarray:
    return np.interp(tvt_values, tw_tvt, tw_vals, left=tw_vals[0], right=tw_vals[-1])


def local_tw_stats(
    centers: np.ndarray, tw_tvt: np.ndarray, tw_gr: np.ndarray, half_window: float
) -> tuple[np.ndarray, np.ndarray]:
    n = len(centers)
    means = np.empty(n, dtype=float)
    stds = np.empty(n, dtype=float)
    left = np.searchsorted(tw_tvt, centers - half_window, side="left")
    right = np.searchsorted(tw_tvt, centers + half_window, side="right")
    for i in range(n):
        seg = tw_gr[left[i] : right[i]]
        if len(seg) < 2:
            means[i] = float(np.interp(centers[i], tw_tvt, tw_gr))
            stds[i] = 0.0
        else:
            means[i] = float(seg.mean())
            stds[i] = float(seg.std())
    return means, stds


def eval_start_index(df: pd.DataFrame) -> int:
    need_pred = df["TVT_input"].isna()
    if need_pred.any():
        return int(need_pred.idxmax())
    if df["GR"].notna().any():
        return int(df["GR"].notna().idxmax())
    return len(df)


def add_typewell_features(df: pd.DataFrame, tw: pd.DataFrame) -> pd.DataFrame:
    tw_tvt = tw["TVT"].to_numpy(dtype=float)
    tw_gr = tw["GR"].to_numpy(dtype=float)
    tw_dgr = np.gradient(tw_gr, tw_tvt)

    tvt_lin = df["tvt_linear"].to_numpy(dtype=float)
    tvt_anc = df["tvt_anchor"].to_numpy(dtype=float)

    df["tw_gr_at_linear"] = interp_tw(tvt_lin, tw_tvt, tw_gr)
    df["tw_gr_at_anchor"] = interp_tw(tvt_anc, tw_tvt, tw_gr)
    df["tw_dgr_at_linear"] = interp_tw(tvt_lin, tw_tvt, tw_dgr)
    df["gr_minus_tw"] = df["GR"].astype(float).ffill() - df["tw_gr_at_linear"]

    for hw in TW_LOCAL_WINDOWS:
        m, s = local_tw_stats(tvt_lin, tw_tvt, tw_gr, hw / 2.0)
        df[f"tw_local_mean_{hw}"] = m
        df[f"tw_local_std_{hw}"] = s
        df[f"gr_minus_tw_local_{hw}"] = df["GR"].astype(float).ffill() - m

    return df


def build_features(h: pd.DataFrame, tw: pd.DataFrame, simulate_eval: bool = False) -> pd.DataFrame:
    df = h.copy()
    eval_start = eval_start_index(df)
    df["eval_start"] = eval_start
    df["in_eval"] = df["row_idx"] >= eval_start

    tvt_in = df["TVT_input"].copy()
    if simulate_eval and not tvt_in.isna().all():
        tvt_in.iloc[eval_start:] = np.nan

    prev_tvt = tvt_in.shift(1)
    df["tvt_anchor"] = prev_tvt.ffill()
    anchor_md = df["MD"].shift(1).where(prev_tvt.notna()).ffill()
    df["md_anchor"] = anchor_md
    df["md_since_anchor"] = df["MD"] - df["md_anchor"]
    df["tvt_linear"] = df["tvt_anchor"] + df["md_since_anchor"]
    df["ft_into_eval"] = np.maximum(0, df["row_idx"] - eval_start)

    df["d_md"] = df["MD"].diff().fillna(0)
    df["d_z"] = df["Z"].diff().fillna(0)

    gr = df["GR"].astype(float)
    gr_filled = gr.ffill()
    for w in ROLL_WINDOWS:
        df[f"gr_roll_mean_{w}"] = gr_filled.rolling(w, min_periods=1).mean()
        df[f"gr_roll_std_{w}"] = gr_filled.rolling(w, min_periods=1).std().fillna(0)
    df["gr_missing"] = gr.isna().astype(np.int8)

    df = add_typewell_features(df, tw)

    if "TVT" in df.columns:
        df["delta_tvt"] = df["TVT"] - df.groupby("well")["TVT"].shift(1)

    return df


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "d_md", "d_z",
    "tvt_anchor", "md_since_anchor", "tvt_linear", "ft_into_eval",
    "GR", "gr_missing",
    "tw_gr_at_linear", "tw_gr_at_anchor", "tw_dgr_at_linear", "gr_minus_tw",
    *[f"tw_local_mean_{w}" for w in TW_LOCAL_WINDOWS],
    *[f"tw_local_std_{w}" for w in TW_LOCAL_WINDOWS],
    *[f"gr_minus_tw_local_{w}" for w in TW_LOCAL_WINDOWS],
    *[f"gr_roll_mean_{w}" for w in ROLL_WINDOWS],
    *[f"gr_roll_std_{w}" for w in ROLL_WINDOWS],
]


def featurize_well(well: str, data_dir: Path, simulate_eval: bool = False) -> pd.DataFrame:
    h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)
    return build_features(h, tw, simulate_eval=simulate_eval)


def deltas_to_tvt(sub: pd.DataFrame, delta_pred: np.ndarray) -> np.ndarray:
    sub = sub.sort_values("row_idx")
    anchor = sub["tvt_anchor"].iloc[0]
    return anchor + np.cumsum(delta_pred)


def train_lgb(X_tr, y_tr, X_va, y_va):
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)],
    )
    return model, model.best_iteration


def train_xgb(X_tr, y_tr, X_va, y_va):
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_va, label=y_va)
    model = xgb.train(
        XGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dvalid, "valid")],
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False,
    )
    return model, model.best_iteration


def train_cat(X_tr, y_tr, X_va, y_va):
    train_pool = Pool(X_tr, y_tr)
    valid_pool = Pool(X_va, y_va)
    model = CatBoostRegressor(**CAT_PARAMS, iterations=NUM_BOOST_ROUND)
    model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=EARLY_STOPPING, use_best_model=True)
    return model, model.get_best_iteration()


def predict_delta(model_name: str, model, X: pd.DataFrame, best_iter: int) -> np.ndarray:
    if model_name == "lgb":
        return model.predict(X, num_iteration=best_iter)
    if model_name == "xgb":
        return model.predict(xgb.DMatrix(X), iteration_range=(0, best_iter + 1))
    return model.predict(Pool(X), ntree_end=best_iter + 1)


def fit_full(model_name: str, X: pd.DataFrame, y: pd.Series, best_iter: int):
    n = max(best_iter, 1)
    if model_name == "lgb":
        return lgb.train(LGB_PARAMS, lgb.Dataset(X, label=y), num_boost_round=n)
    if model_name == "xgb":
        return xgb.train(XGB_PARAMS, xgb.DMatrix(X, label=y), num_boost_round=n, verbose_eval=False)
    model = CatBoostRegressor(**CAT_PARAMS, iterations=n)
    model.fit(Pool(X, y), verbose=False)
    return model


def predict_delta_full(model_name: str, model, X: pd.DataFrame) -> np.ndarray:
    if model_name == "lgb":
        return model.predict(X)
    if model_name == "xgb":
        return model.predict(xgb.DMatrix(X))
    return model.predict(Pool(X))

In [ ]:
def build_training_frame(wells: list[str], data_dir: Path) -> pd.DataFrame:
    parts = []
    for well in wells:
        df = featurize_well(well, data_dir, simulate_eval=True)
        if "TVT" not in df.columns:
            continue
        mask = (
            df["in_eval"]
            & df["GR"].notna()
            & df["TVT"].notna()
            & df["tvt_anchor"].notna()
            & df["delta_tvt"].notna()
        )
        parts.append(df.loc[mask])
    return pd.concat(parts, ignore_index=True)


train_wells = list_wells(TRAIN_DIR)
print(f"Training wells: {len(train_wells)}")

train_df = build_training_frame(train_wells, TRAIN_DIR)
print(f"Training rows (eval zone only): {len(train_df):,}")
print(train_df[["TVT", "delta_tvt", "tvt_linear", "GR"]].describe().round(3))

In [ ]:
X = train_df[FEATURE_COLS]
y = train_df["delta_tvt"].astype(float)
groups = train_df["well"]

oof_tvt = np.zeros(len(train_df))
best_iters = {name: [] for name in MODEL_NAMES}

gkf = GroupKFold(n_splits=N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    fold_delta = np.zeros(len(va_idx))
    for name in MODEL_NAMES:
        if name == "lgb":
            model, it = train_lgb(X_tr, y_tr, X_va, y_va)
        elif name == "xgb":
            model, it = train_xgb(X_tr, y_tr, X_va, y_va)
        else:
            model, it = train_cat(X_tr, y_tr, X_va, y_va)
        best_iters[name].append(it)
        fold_delta += predict_delta(name, model, X_va, it)
    fold_delta /= len(MODEL_NAMES)

    va = train_df.iloc[va_idx]
    for well in va["well"].unique():
        well_mask = (va["well"] == well).to_numpy()
        wpos = va_idx[well_mask]
        oof_tvt[wpos] = deltas_to_tvt(train_df.iloc[wpos], fold_delta[well_mask])

    fold_rmse = np.sqrt(mean_squared_error(train_df.iloc[va_idx]["TVT"], oof_tvt[va_idx]))
    print(f"Fold {fold + 1}/{N_FOLDS}  ensemble TVT RMSE = {fold_rmse:.4f}")

cv_rmse = np.sqrt(mean_squared_error(train_df["TVT"], oof_tvt))
print(f"\nOOF eval-zone TVT RMSE (ensemble): {cv_rmse:.4f}")

final_iters = {name: max(int(np.mean(best_iters[name])), 1) for name in MODEL_NAMES}
print("Full-train iterations:", final_iters)

final_models = {}
for name in MODEL_NAMES:
    final_models[name] = fit_full(name, X, y, final_iters[name])
    print(f"Fitted full {name}")

In [ ]:
def predict_eval_zone(well: str, data_dir: Path, final_models: dict) -> pd.DataFrame:
    h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)
    df = build_features(h, tw, simulate_eval=False)
    sub = df.loc[df["in_eval"] & h["TVT_input"].isna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=["id", "tvt"])

    Xp = sub[FEATURE_COLS]
    delta_pred = np.zeros(len(sub))
    for name in MODEL_NAMES:
        delta_pred += predict_delta_full(name, final_models[name], Xp)
    delta_pred /= len(MODEL_NAMES)

    sub["tvt"] = deltas_to_tvt(sub, delta_pred)
    sub["id"] = sub["well"] + "_" + sub["row_idx"].astype(str)
    return sub[["id", "tvt"]]


test_wells = list_wells(TEST_DIR)
print(f"Test wells: {len(test_wells)}")

pred_parts = [predict_eval_zone(w, TEST_DIR, final_models) for w in test_wells]
submission = pd.concat(pred_parts, ignore_index=True)

sample = pd.read_csv(SAMPLE_SUB)
submission = sample[["id"]].merge(submission, on="id", how="left")
missing = submission["tvt"].isna().sum()
if missing:
    raise ValueError(f"Missing predictions for {missing} sample ids")

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nWrote submission.csv  rows={len(submission):,}")
print(f"tvt range: {submission['tvt'].min():.2f} .. {submission['tvt'].max():.2f}")